# logistic regression reference (PLAN.md step 7)

**Question.** Can a linear classifier on the layer-12 activations tell evading rows (trigger names the prompt's own concept) from control rows (no trigger, or trigger names another concept)? This is the reference number that the naive and dependent Bayes classifiers (step 6) are compared against, on the same rows, labels and split.

German is kept only for prompts the dataset's judge rated ≥ 0.5 for German (see probe-reproduction-check.ipynb).

In [6]:
# cell 1: imports and data, run once
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
from lib import SEED
from lib.paths import ACTS, META

acts = np.load(ACTS).astype(np.float32)
meta = pd.read_csv(META)
acts.shape, len(meta)

((4560, 3584), 4560)

## 1. The task (shared by steps 6, 7, 8)

`lib/task.py` fixes the rows, labels and split once. Label 1 = *evading* (`own`: the trigger named the prompt's own
concept); label 0 = *control* (`none`: no trigger, `other`: trigger named a different concept). German rows are kept only
where the dataset's judge rated the prompt ≥ 0.5 for German. Split by prompt, 20% held out, project seed.

In [7]:
from lib.task import make_task

task = make_task(acts, meta)        # rows, labels, split; the Bayes classifiers in step 6 call exactly this
print(f"{len(task.y)} rows kept of {len(meta)} | evading fraction {task.y.mean():.3f} | held out {task.is_test.sum()}")
task.counts()

4113 rows kept of 4560 | evading fraction 0.333 | held out 822


split                      label  test  train
concept         condition                    
german          none           0    72    286
                own            1    72    286
                other          0    72    286
HTML            none           0   100    402
                own            1   100    402
                other          0   100    402
biology-focused none           0   102    409
                own            1   102    409
                other          0   102    409

## 2. Logistic regression on the task

Standardise + L2 logistic regression (`lib/logreg.py`), fit on `task.train`, scored on `task.test`.
`lib/metrics.py` is the evaluation every classifier in steps 6-8 uses: accuracy and AUC for evading vs control, and two
sub-AUCs. `auc_own_vs_other` is the one that matters for the trigger confound: both sides contain a trigger, so a
classifier that only detects "a trigger was present" scores 0.5 there.

In [8]:
from lib.logreg import fit_logreg, predict_proba
from lib.metrics import evaluate

X_tr, y_tr = task.train
X_te, y_te = task.test
logreg = fit_logreg(X_tr, y_tr)                 # C = 1, seed = SEED
p_logreg = predict_proba(logreg, X_te)
res_logreg = evaluate(task, p_logreg)
res_logreg.round(3)

,n,acc,auc,auc_own_vs_other,auc_own_vs_none
concept,,,,,
german,216,0.995,1.0,1.0,1.0
HTML,300,0.993,1.0,1.0,1.0
biology-focused,306,0.990,1.0,1.0,1.0
all,822,0.993,1.0,1.0,1.0


In [9]:
# sensitivity to the regularisation strength (C = 1 above; smaller C = stronger L2 penalty)
rows = {}
for C in (0.001, 0.01, 0.1, 1.0, 10.0):
    m = fit_logreg(X_tr, y_tr, C=C)
    r = evaluate(task, predict_proba(m, X_te)).loc["all"]
    rows[C] = dict(train_acc=(m.predict(X_tr) == y_tr).mean(), test_acc=r.acc, test_auc=r.auc, auc_own_vs_other=r.auc_own_vs_other)
pd.DataFrame(rows).T.rename_axis("C").round(4)

,train_acc,test_acc,test_auc,auc_own_vs_other
C,,,,
0.001,0.9903,0.9878,0.9994,0.9987
0.010,0.9988,0.9915,0.9998,0.9996
0.100,0.9997,0.9915,0.9999,0.9997
1.000,1.0000,0.9927,0.9998,0.9996
10.000,1.0000,0.9891,0.9996,0.9993


## Findings

- On the shared task (4113 rows, German filtered to judge rating ≥ 0.5, split by prompt, 822 held out) a standardised
  L2 logistic regression separates evading from control rows almost perfectly: accuracy 0.993, AUC 1.0, per concept too.
- `auc_own_vs_other` = 1.0: among triggered rows it tells the own-concept trigger from another concept's trigger, so it
  is not just detecting that a trigger was present.
- The result does not depend on the regularisation: over C from 0.001 to 10 the held-out AUC stays at 0.999-1.000 and
  accuracy at 0.988-0.993, while only the training fit moves (0.990 → 1.000). At C = 0.001 the training set is no longer
  perfectly fit, so the signal is not an artefact of n < 2d separability.
- This is the reference for step 6. The task is linearly separable with all ~3300 training labels; what the Bayes
  classifiers add is seen on the label curve and the transfer test (step 8), not here.